# Robot Arena
Configura tu robot en **Mi robot** y enfréntate a los jefes en **Combates**. Cada jefe tiene dos formas de jugar:

- `entrenar(N)`: juegas tú con los botones. No da contraseña, pero al acabar te enseña tu combate convertido en código.
- `luchar(N)`: el combate definitivo. Lucha el código de tu robot y, si gana, te da la contraseña.

Las celdas se ejecutan con **Mayús + Intro**. Cada vez que cambies algo en Mi robot, vuelve a ejecutar esa celda.

## Paso 0 · Motor del juego

In [ ]:
# @title Motor del juego: ejecútalo y no lo toques { display-mode: "form" }
%pip install -q ipywidgets
import random
from types import SimpleNamespace

_rng = random.Random()   # generador propio: cambiar random.randint desde fuera no afecta a los combates
NIVEL_OMEGA = 7
NIVEL_SECRETO = 8

_DATOS = "==Qf91XZ1JHdgojIl5Wdt5WaiACLxAiOiEWayVnZiACLwAiOiIXZzFGbiACLw4CMgojIzFmbpB3clJCIsADI6IibldWZyJCIs0FNxACL4sFI6ISZwx2bnJCIsADI6ISYyVHZh1mchJCIsATNxAiOiEGZpZnIgwiIPR1QFRVSVFlUBBCTFJCI6ISZyJWbv5mI7BiOigjIgwSfxAiOiEWayVnZiACL1UDI6IiclNXYsJCIsAjLwAiOiMXYulGczVmIgwCMgojIuV2ZlJnIgwSXwIDIsQTMbBiOiUGcs92ZiACLwQDI6ISYyVHZh1mchJCIsAjNyAiOiEGZpZnIgwiIBdURN9kIgojIlJnYt9mbisHI6IyNiACL9VWdyRHI6ICZhRWaslmYhh2Xvx2bzJCIsUWdyRHI6IybqVGczVmIgwyMgojIhlmc1ZmIgwCOzAiOiIXZzFGbiACLw4SMgojIzFmbpB3clJCIsADI6IibldWZyJCIs0FOxACLyEzWgojIlBHbvdmIgwCMzAiOiEmc1RWYtJXYiACLwAjMgojIhRWa2JCIsIyTKVEUTVkIgojIlJnYt9mbisHI6IiNiACL9VWdyRHI6Iycl52bpNmblRnbpJCIsUWdyRHI6IybkFGZulGbiJCIsADI6ISYpJXdmJCIsADI6IiclNXYsJCIsAjLwAiOiMXYulGczVmIgwCOgojIuV2ZlJnIgwSX4ACL0sFI6ISZwx2bnJCIsUDNgojIhJXdkFWbyFmIgwCMwIDI6ISYklmdiACLi4UgDTVSUJCI6ISZyJWbv5mI7BiOiUjIgwSfwYDI6IybqlmZf9GZhNXZwJCIsADI6ISZyJWdj9FciACLz4CMgojIvRWYzVGcfBnIgwSZ1JHdgojIzVmbvl2YuVGdulmIgwCMgojIhlmc1ZmIgwCMgojIyV2chxmIgwCMuADI6Iych5WawNXZiACLwAiOi4WZnVmciACLdRDIsIzWgojIlBHbvdmIgwCMyAiOiEmc1RWYtJXYiACLwUzMgojIhRWa2JCIsIyTMxUSUJVQNJCI6ISZyJWbv5mI7BiOiQjIgwSfxAiOiEWayVnZiACLwAiOiIXZzFGbiACLw4CMgojIzFmbpB3clJCIsADI6IibldWZyJCIs0lNgwyMbBiOiUGcs92ZiACL1QDI6ISYyVHZh1mchJCIsATNxAiOiEGZpZnIgwiITl0UF1UiD7kIgojIlJnYt9mbisHI6IyMiACL9BDI6ISYpJXdmJCIsADI6IiclNXYsJCIsAjLwAiOiMXYulGczVmIgwCMgojIuV2ZlJnIgwSXwEDIsUzWgojIlBHbvdmIgwCMgojIhJXdkFWbyFmIgwCMyEDI6ISYklmdiACLiEkUSFEVBh0QiAiOiUmci12buJyegojIyICIs0HMgojIhlmc1ZmIgwCMgojIyV2chxmIgwCMuADI6Iych5WawNXZiACLwAiOi4WZnVmciACLdBTMgwSNbBiOiUGcs92ZiACLwAiOiEmc1RWYtJXYiACLwITMgojIhRWa2JCIsISQSJVQUFESDJCI6ISZyJWbv5mI7BiOiEjI7BiOiMXZmVmaiACL911NgwiNbBiOiQUQElETJJUQI91UFxURWlkTiACL1AiOi80QJRVSSN0XMVkVJ5kIgwSNyAiOiI1TMF0QiACLwQDI6ICWB10XT9kTSVFViACL1AiOi8ERBNVRQJCIsADNgojIPRlUFlkQVNkIgwCM1EDI6ICTBR1TUJCIsAzNgojIUFEVT9FWB1kIgwCM1AiOiEUSHJVROVkI7BiOiMXYsdWZyJye"


def _reglas(_d=_DATOS):
    import json, base64
    return json.loads(base64.b64decode(_d[::-1]).decode())["reglas"]


def _jefe(nivel, _d=_DATOS):
    import json, base64
    return json.loads(base64.b64decode(_d[::-1]).decode())["jefes"][str(nivel)]


class Trampa(Exception):
    pass


def validar_stats(vida, ataque, curacion):
    R = _reglas()
    TOTAL, MAX_STAT = R["TOTAL"], R["MAX_STAT"]
    for x in (vida, ataque, curacion):
        if type(x) is not int:
            raise Trampa("Usa números enteros (sin decimales ni comillas).")
    if vida + ataque + curacion != TOTAL:
        raise Trampa(f"Tus atributos suman {vida + ataque + curacion}. Deben sumar exactamente {TOTAL}.")
    if max(vida, ataque, curacion) > MAX_STAT:
        raise Trampa(f"Ningún atributo puede pasar de {MAX_STAT}.")


class Combate:
    def __init__(self, vida, ataque, curacion, nivel, regla=None, habilidad=None):
        validar_stats(vida, ataque, curacion)
        self.nivel = nivel
        self._r = _reglas()
        self.jefe = _jefe(nivel)
        self.maxvida, self.ataque, self.curacion = vida, ataque, curacion
        self.regla = regla if nivel >= self._r["NIVEL_CRITICO"] else None
        self.habilidad = habilidad if nivel in self._r["NIVELES_HABILIDAD"] else None
        self.reiniciar()

    # --- estado -------------------------------------------------------
    def reiniciar(self):
        self.vida = self.maxvida
        self.jefe_max = self.jefe["vida"]
        self.jefe_vida = self.jefe_max
        self.turno = 0
        self.turbo = True
        self.cargando = False
        self.intencion = "normal"   # movimiento anunciado del jefe: normal, pesado o cubre
        self.anulada = False
        self.eventos = []           # lo que ha pasado en el último turno: (quién, texto)
        self.historial = []         # (turno, eventos) de todos los turnos
        self._sello = self._foto()

    def _foto(self):
        return (self.vida, self.maxvida, self.ataque, self.curacion, self.jefe_vida, self.turbo, self.nivel)

    @property
    def mensaje(self):
        return " ".join(t for _, t in self.eventos)

    @property
    def gana(self):
        return self.jefe_vida <= 0 and self.vida > 0 and not self.anulada

    @property
    def terminado(self):
        return self.anulada or self.vida <= 0 or self.jefe_vida <= 0

    def fase2(self):
        return self.nivel == NIVEL_OMEGA and self.jefe_vida <= self.jefe_max // 2

    def aviso(self):
        if self.cargando:
            return "laser"
        return {"pesado": "pesado", "cubre": "cubre"}.get(self.intencion, "nada")

    def info(self):
        """Lo que ven las funciones del alumno: una copia, para que no puedan cambiar la partida."""
        return SimpleNamespace(vida=self.vida, vida_max=self.maxvida, vida_jefe=self.jefe_vida,
                               vida_jefe_max=self.jefe_max, jefe=self.jefe["nombre"], aviso=self.aviso(),
                               turbo=self.turbo, fase=2 if self.fase2() else 1, turno=self.turno + 1)

    def armadura(self):
        return self.jefe["armadura"] + (self._r["CUBIERTO"] if self.intencion == "cubre" else 0)

    def espinas(self):
        return 1.0 if self.fase2() else self.jefe["espinas"]

    # --- turno --------------------------------------------------------
    def actuar(self, accion):
        turno = self.turno
        self.eventos = []
        self._actuar(accion)
        if self.turno != turno or self.anulada:
            self.historial.append((self.turno, list(self.eventos)))

    def _yo(self, t):
        self.eventos.append(("yo", t))

    def _el(self, t):
        self.eventos.append(("jefe", t))

    def _aviso(self, t):
        self.eventos.append(("aviso", t))

    def _actuar(self, accion):
        if self.terminado:
            return
        if self._foto() != self._sello:
            self.anulada = True
            self._aviso("ANTITRAMPAS: alguien ha tocado la vida o los atributos a mano. Partida anulada.")
            return
        validas = ["atacar", "curar", "escudo", "turbo"] + (["habilidad"] if self.habilidad else [])
        if self.jefe.get("solo_habilidad") and self.habilidad:
            accion = "habilidad"      # contra ESPEJO solo funciona la habilidad
        if accion not in validas or (accion == "turbo" and not self.turbo):
            return
        self.turno += 1
        escudo = False
        j = self.jefe

        if accion in ("atacar", "turbo"):
            critico = blindado = False
            if accion == "turbo":
                base = 2 * self.ataque
                self.turbo = False
            else:
                base = self.ataque
                if self.regla is not None:
                    base = int(self.regla(self.ataque, self.info()))
                    if base > self.ataque:   # golpe crítico: sobrecalienta tu robot
                        critico = True
                        self.vida = max(0, self.vida - self._r["CALOR"])
                        if j.get("blindado") and self.jefe_vida > self.jefe_max // 2:
                            base = self.ataque
                            blindado = True
                    if self.nivel >= self._r["NIVELES_HABILIDAD"][0] and base > 2 * self.ataque:
                        self.vida = 0
                        self._aviso(f"ANTITRAMPAS: golpe ilegal de {base} (máximo {2 * self.ataque}). "
                                    f"{j['nombre']} te lo devuelve.")
                        self._sello = self._foto()
                        return
            arm, esp = self.armadura(), self.espinas()
            real = 0 if j.get("inmune") else max(0, base - arm)
            self.jefe_vida = max(0, self.jefe_vida - real)
            nombre = "Turbo" if accion == "turbo" else ("Crítico" if critico else "Atacas")
            if j.get("inmune"):
                self._yo(f"{nombre}: 0 de daño. {j['nombre']} es inmune: esa acción la programó él.")
            elif arm:
                self._yo(f"{nombre}: {base} − {arm} de armadura = {real} de daño.")
            else:
                self._yo(f"{nombre}: {real} de daño.")
            if critico:
                self._yo(f"Te sobrecalientas: −{self._r['CALOR']} de vida." +
                         (" El núcleo blindado ha parado el crítico." if blindado else ""))
            rebote = int(real * esp)
            if rebote:
                self.vida = max(0, self.vida - rebote)
                self._el(f"Te devuelve el golpe: −{rebote} de vida.")
        elif accion == "curar":
            antes = self.vida
            self.vida = min(self.maxvida, self.vida + self.curacion)
            self._yo(f"Te curas: +{self.vida - antes} de vida.")
        elif accion == "escudo":
            escudo = True
            self._yo("Escudo: este turno recibes un cuarto del daño.")
        else:  # habilidad
            d, h = self.habilidad(self.info())
            if type(d) is not int or type(h) is not int or d < 0 or h < 0:
                self.anulada = True
                self._aviso("ANTITRAMPAS: la habilidad debe devolver dos números enteros positivos. Partida anulada.")
                return
            if d + h > self._r["ENERGIA"]:
                self.anulada = True
                self._aviso(f"ANTITRAMPAS: tu habilidad usa {d + h} de energía (máximo {self._r['ENERGIA']}). Partida anulada.")
                return
            antes = self.vida
            self.vida = min(self.maxvida, self.vida + h)
            curado = self.vida - antes
            if self.nivel == NIVEL_OMEGA and not self.fase2() and d:
                self._yo(f"Habilidad: OMEGA absorbe tu energía (0 de daño), +{curado} de vida.")
                d = 0
            elif j.get("espejo") and d and not self.cargando:
                self.vida = max(0, self.vida - d)
                self._yo(f"Habilidad: +{curado} de vida, pero ESPEJO refleja tus {d} de daño.")
                self._el(f"Te devuelve la energía: −{d} de vida.")
                d = 0
            elif j.get("espejo") and d:
                d *= 2
                self._yo(f"Habilidad: núcleo expuesto, {d} de daño (doble), +{curado} de vida.")
            else:
                self._yo(f"Habilidad: {d} de daño directo, +{curado} de vida.")
            self.jefe_vida = max(0, self.jefe_vida - d)

        if self.vida <= 0:
            self._aviso("Tu robot se ha destruido. Fin de partida.")
        elif self.jefe_vida <= 0:
            self._aviso("¡VICTORIA!")
        else:
            self._turno_jefe(escudo)
            if self.vida <= 0:
                self._aviso("Tu robot se ha destruido. Fin de partida.")
            elif self.turno >= self._r["TURNOS_MAX"]:
                self.vida = 0
                self._aviso(f"Han pasado {self._r['TURNOS_MAX']} turnos: tu robot se queda sin batería. Fin de partida.")
        self._sello = self._foto()

    def _turno_jefe(self, escudo):
        j = self.jefe
        if j["regen"]:
            self.jefe_vida = min(self.jefe_max, self.jefe_vida + j["regen"])
            self._el(f"Se repara: +{j['regen']} de vida.")
        if self.cargando:
            golpe, nombre = j["laser"], "Dispara el LÁSER"
            self.cargando = False
        elif j["laser"] and self.turno % 3 == 2:
            self.cargando = True
            self._aviso(f"{j['nombre']} está cargando el LÁSER: dispara el próximo turno.")
            return
        else:
            golpe = _rng.randint(*j["golpe"]) + int(j["furia"] * (self.turno - 1))
            nombre = "Te golpea"
            if self.intencion == "pesado":
                golpe, nombre = j.get("pesado_fijo", int(golpe * self._r["PESADO"])), "GOLPE PESADO"
            elif self.intencion == "cubre":
                golpe, nombre = golpe // 2, "Golpe flojo (estaba cubierto)"
        if escudo:
            golpe //= 4
        self.vida = max(0, self.vida - golpe)
        self._el(f"{nombre}: −{golpe} de vida" + (" (tu escudo para el resto)." if escudo else "."))
        if j.get("intenciones") and self.vida > 0:
            r = _rng.random()
            pp, pc = j.get("p_pesado", 0.25), j.get("p_cubre", 0.20)
            self.intencion = "pesado" if r < pp else ("cubre" if r < pp + pc else "normal")
            if self.intencion == "pesado":
                self._aviso(f"{j['nombre']} prepara un GOLPE PESADO para el próximo turno.")
            elif self.intencion == "cubre":
                self._aviso(f"{j['nombre']} se cubre: el próximo turno su armadura sube {self._r['CUBIERTO']}.")
        else:
            self.intencion = "normal"


# ======================= PARTE GRÁFICA =======================
FUENTE = "Verdana, sans-serif"   # fuente del juego: cambia aquí el nombre (sin comillas simples)
import base64
import re
from html import escape
import ipywidgets as w
from IPython.display import display, HTML

COLORES = {"azul": "#38bdf8", "rojo": "#fb7185", "verde": "#34d399", "morado": "#c084fc",
           "amarillo": "#facc15", "rosa": "#f472b6", "naranja": "#fb923c",
           "blanco": "#f8fafc", "gris": "#94a3b8", "cian": "#22d3ee"}
COLOR_JEFE = {1: "#94a3b8", 2: "#94a3b8", 3: "#fb7185", 4: "#fb923c", 5: "#f59e0b", 6: "#a5f3fc", 7: "#a78bfa", 8: "#e2e8f0"}
CABEZAS = ("cuadrada", "redonda", "triangular")
ANTENAS = ("antena", "orejas", "hélice", "nada")
EMBLEMAS = ("rayo", "estrella", "corazón", "inicial")
_PUERTA_ABIERTA = False

_K = {1: "QUNSRVVU", 2: "QUNSRVVU", 3: "T0lUTE9W", 4: "RUpBTkFSR05F", 5: "QVBTSUhD", 6: "T1RJVUNSSUM=", 7: "QVZPTlJFUFVT", 8: "QVJFU0FSVEFUUkVVUA=="}
def _clave(n):
    return base64.b64decode(_K[n]).decode()[::-1]

FRASES = {
    1: ["*clonk*", "Bzzt… ¿quién me ha encendido?", "Soy viejo, pero muy oxidado."],
    2: ["¿Otra vez tú?", "*clonk* *clonk*", "Esta vez no me pillas… creo."],
    3: ["Mi armadura es impenetrable, humano.", "¿Eso era un ataque?", "Ni me he despeinado."],
    4: ["¡Prepárate para el martillazo!", "Demoler es mi especialidad.", "*BOOM*"],
    5: ["Me reparo más rápido de lo que me rompes.", "¡Tornillos nuevos, gracias!", "Soy eterno."],
    6: ["Todo lo que me das, te lo devuelvo.", "Mírate en mi reflejo…", "¿Seguro que quieres atacarme?"],
    7: ["Soy el último nivel.", "Ningún humano pulsa botones tan rápido como yo pienso.", "Programa o perece."],
    8: ["Yo escribí este mundo.", "Todo lo que sabes hacer, lo programé yo.",
        "Solo puede vencerme una acción que mi creador no programó."],
}

CSS = """<style>
@keyframes ra-shake{0%,100%{transform:translateX(0)}20%{transform:translateX(-8px)}40%{transform:translateX(8px)}60%{transform:translateX(-5px)}80%{transform:translateX(5px)}}
@keyframes ra-pop{0%{transform:scale(.5);opacity:0}70%{transform:scale(1.08)}100%{transform:scale(1);opacity:1}}
.ra-hit{animation:ra-shake .4s 1}
.ra-pop{animation:ra-pop .5s}
@keyframes ra-float{0%{opacity:0;transform:translateY(12px)}15%{opacity:1}100%{opacity:0;transform:translateY(-45px)}}
.ra-float{position:absolute;top:35px;left:0;right:0;text-align:center;font-weight:bold;font-size:30px;
  text-shadow:0 2px 4px #000;animation:ra-float 1.6s forwards;pointer-events:none}
.ra-lin{display:flex;gap:10px;padding:3px 0}
.ra-lab{min-width:110px;font-weight:bold}
</style>"""

def _color(c):
    if isinstance(c, str) and c in COLORES:
        return COLORES[c]
    if isinstance(c, str) and re.fullmatch(r"#[0-9a-fA-F]{6}", c):
        return c
    return None


def robot_svg(color, jefe=False, ojos=None, cabeza="cuadrada", antena="antena", emblema="rayo", inicial="?"):
    ojos = ojos or color
    if jefe:
        ojos_svg = f"<path d='M58 44L73 50L73 56L58 56Z' fill='{color}'/><path d='M102 44L87 50L87 56L102 56Z' fill='{color}'/>"
        arriba = f"<path d='M45 31L36 8L58 29Z' fill='{color}'/><path d='M115 31L124 8L102 29Z' fill='{color}'/>"
        pecho = "<circle cx='80' cy='97' r='9' fill='#0f172a'/><circle cx='80' cy='97' r='4' fill='#fff'/>"
        cabeza_svg = f"<rect x='43' y='29' width='74' height='45' rx='9' fill='{color}'/>"
    else:
        ojos_svg = (f"<rect x='60' y='44' width='13' height='12' rx='3' fill='{ojos}'/>"
                    f"<rect x='87' y='44' width='13' height='12' rx='3' fill='{ojos}'/>")
        arriba = {
            "antena": f"<path d='M80 4V30' stroke='#94a3b8' stroke-width='5'/><circle cx='80' cy='6' r='6' fill='{color}'/>",
            "orejas": f"<rect x='33' y='38' width='12' height='26' rx='4' fill='{color}'/><rect x='115' y='38' width='12' height='26' rx='4' fill='{color}'/>",
            "hélice": (f"<path d='M80 8V30' stroke='#94a3b8' stroke-width='5'/>"
                       f"<ellipse cx='80' cy='8' rx='26' ry='5' fill='{color}'/><circle cx='80' cy='8' r='4' fill='#e2e8f0'/>"),
            "nada": "",
        }.get(antena, "")
        cabeza_svg = {
            "cuadrada": f"<rect x='43' y='29' width='74' height='45' rx='9' fill='{color}'/>",
            "redonda": f"<rect x='41' y='24' width='78' height='52' rx='26' fill='{color}'/>",
            "triangular": f"<path d='M80 10L121 42V74H39V42Z' fill='{color}'/>",
        }.get(cabeza, "")
        pecho = {
            "rayo": "<path d='M82 84L70 99H79L76 111L91 94H82Z' fill='#fff'/>",
            "estrella": "<path d='M80 84L83.5 93H93L85.5 98.5L88.5 108L80 102L71.5 108L74.5 98.5L67 93H76.5Z' fill='#fff'/>",
            "corazón": "<path d='M80 109C70 101 66 96 66 91C66 87 69 84 73 84C76 84 78 86 80 88C82 86 84 84 87 84C91 84 94 87 94 91C94 96 90 101 80 109Z' fill='#fff'/>",
            "inicial": (f"<text x='80' y='105' text-anchor='middle' font-size='22' font-weight='bold' "
                        f"font-family='Arial' fill='#fff'>{escape(inicial)}</text>"),
        }.get(emblema, "")
    return (f"<svg viewBox='0 0 160 145' width='140' height='127' xmlns='http://www.w3.org/2000/svg'>"
            f"<ellipse cx='80' cy='136' rx='45' ry='6' fill='#080e1a'/>{arriba}{cabeza_svg}"
            f"<rect x='51' y='37' width='58' height='27' rx='4' fill='#0f172a'/>{ojos_svg}"
            f"<rect x='53' y='79' width='54' height='38' rx='6' fill='{color}'/>"
            f"<rect x='30' y='79' width='18' height='34' rx='5' fill='#64748b'/><rect x='112' y='79' width='18' height='34' rx='5' fill='#64748b'/>"
            f"<rect x='54' y='120' width='22' height='13' rx='4' fill='#94a3b8'/><rect x='84' y='120' width='22' height='13' rx='4' fill='#94a3b8'/>"
            f"{pecho}</svg>")



def _aviso(texto, color="#7f1d1d"):
    display(HTML(f"<div style='max-width:640px;background:{color};color:#fff;padding:14px;border-radius:12px;"
                 f"font-family:{FUENTE};font-size:15px'>{texto}</div>"))


ESTILO_DE_SERIE = {"ojos": "blanco", "cabeza": "cuadrada", "antena": "antena", "emblema": "rayo", "frases": []}


def _preparar(nombre, color, vida, ataque, curacion, nivel, regla, habilidad, estilo):
    if _color(color) is None or _color(estilo["ojos"]) is None:
        _aviso("Ese color no existe. Elige uno de estos: " + ", ".join(COLORES) +
               ", o escríbelo en hexadecimal entre comillas, por ejemplo \"#ff8800\".")
        return None
    for valor, opciones, que in ((estilo["cabeza"], CABEZAS, "cabeza"), (estilo["antena"], ANTENAS, "antena"),
                                 (estilo["emblema"], EMBLEMAS, "emblema")):
        if valor not in opciones:
            _aviso(f"Esa {que} no existe. Elige una de estas: " + ", ".join(opciones))
            return None
    frases = estilo["frases"]
    if not isinstance(frases, list) or not all(isinstance(f, str) for f in frases):
        _aviso('frases tiene que ser una lista de textos, entre corchetes y separados por comas: '
               '["¡Hola!", "¡Toma ya!"]')
        return None
    try:
        return Combate(vida, ataque, curacion, nivel, regla, habilidad)
    except Trampa as e:
        _aviso(f"ANTITRAMPAS: {escape(str(e))}")
        return None


def _error_codigo(e, donde):
    if isinstance(e, AttributeError) and "SimpleNamespace" in str(e):
        dato = str(e).split("attribute")[-1].strip(" '")
        return (f"Error en tu código ({donde}): tu robot no sabe qué es «{dato}». "
                "Mira en Mi robot la lista de lo que sabe tu robot y revisa cómo está escrito.")
    return f"Error en tu código ({donde}): {type(e).__name__}: {e}. Revísalo y vuelve a ejecutar Mi robot."


def _cerebro(grab, rival):
    """Convierte las teclas que ha pulsado el alumno en una función decidir."""
    usadas = [g["accion"] for g in grab]
    L = ["def decidir(robot):", f"    # Así jugaste tú contra {rival}:"]
    avisos = []
    for g in grab:
        if g["accion"] == "escudo" and g["aviso"] != "nada" and g["aviso"] not in avisos:
            avisos.append(g["aviso"])
    for av in avisos:
        L += [f'    if robot.aviso == "{av}":', f'        return "escudo"          # te protegiste cuando avisó']
    if "escudo" in usadas and not avisos:
        L += ["    # (usaste el escudo sin que el jefe avisara de nada)"]
    if any(g["accion"] == "habilidad" and g["aviso"] == "laser" for g in grab) and \
            any(g["accion"] != "habilidad" and g["aviso"] != "laser" for g in grab):
        L += ['    if robot.aviso == "laser":', '        return "habilidad"       # usaste la habilidad con el láser cargando']
    curas = [g["vida"] for g in grab if g["accion"] == "curar"]
    if curas:
        umbral = (max(curas) // 5 + 1) * 5
        L += [f"    if robot.vida < {umbral}:", f'        return "curar"           # te curaste cuando te quedaba poca vida']
    if "turbo" in usadas:
        L += ["    if robot.turbo:", '        return "turbo"           # usaste el turbo']
    golpes = [a for a in usadas if a in ("atacar", "habilidad")]
    resto = max(set(golpes), key=golpes.count) if golpes else "atacar"
    L += [f'    return "{resto}"              # el resto del tiempo']
    return "\n".join(L)


def arena(nombre, color, vida, ataque, curacion, nivel=1, regla=None, habilidad=None, estilo=None,
          decidir=None, practica=False, *, _check=None):
    """Abre la arena. Sin decidir se juega con botones; con decidir lucha el código del alumno."""
    estilo = dict(ESTILO_DE_SERIE, **(estilo or {}))
    juego = _preparar(nombre, color, vida, ataque, curacion, nivel, regla, habilidad, estilo)
    if juego is None:
        return
    jefe = juego.jefe
    mi_color = _color(color)
    panel = w.HTML()
    if decidir is None:
        acciones = [("atacar", "Atacar", "danger"), ("curar", "Curar", "success"),
                    ("escudo", "Escudo", "info"), ("turbo", "Turbo", "warning")]
        if juego.habilidad:
            acciones.append(("habilidad", "Habilidad", "primary"))
        if jefe.get("solo_habilidad"):
            acciones = [a for a in acciones if a[0] == "habilidad"]
    else:
        acciones = [("siguiente", "Siguiente turno", "primary")]
    botones = {a: w.Button(description=t, button_style=s, layout=w.Layout(width="150px" if decidir else "122px"))
               for a, t, s in acciones}
    reset = w.Button(description="Nueva partida", layout=w.Layout(width="150px"))
    for b in list(botones.values()) + [reset]:
        try:
            b.style.font_family = FUENTE
        except Exception:
            pass
    estado = {"d_yo": 0, "d_jefe": 0, "extra": [], "frase": ""}
    grab = []     # teclas pulsadas, para convertirlas en código
    mi_svg = robot_svg(mi_color, False, _color(estilo["ojos"]), estilo["cabeza"], estilo["antena"],
                       estilo["emblema"], (str(nombre) or "?")[0].upper())
    jefe_svg = robot_svg(COLOR_JEFE[nivel], True)
    etiquetas = {"yo": (str(nombre), mi_color), "jefe": (jefe["nombre"], "#f87171"), "aviso": ("Aviso", "#fb923c")}

    def ficha(n, c, pv, maxpv, es_jefe, delta):
        clase = "ra-hit" if delta < 0 else ""
        numero = ""
        if delta:
            numero = (f"<div class='ra-float' style='color:{'#4ade80' if delta > 0 else '#f87171'}'>"
                      f"{'+' if delta > 0 else ''}{delta}</div>")
        return (f"<div class='{clase}' style='position:relative;flex:1;min-width:150px;text-align:center;padding:8px'>"
                f"{numero}{jefe_svg if es_jefe else mi_svg}<div style='font-weight:bold;font-size:17px'>{escape(n)}</div>"
                f"<div style='background:#334155;height:13px;border-radius:8px;overflow:hidden;margin:8px 0'>"
                f"<div style='width:{100 * pv / maxpv:.1f}%;height:100%;background:{c}'></div></div>"
                f"<b>{pv}/{maxpv} PV</b></div>")

    def _limpio():
        try:
            validar_stats(juego.maxvida, juego.ataque, juego.curacion)
        except Exception:
            return False
        return bool(_check and _check())

    def lineas(eventos, pequeno=False):
        tam = "12px" if pequeno else "15px"
        html = ""
        for quien, texto in eventos:
            lab, col = etiquetas[quien]
            html += (f"<div class='ra-lin' style='font-size:{tam}'><span class='ra-lab' style='color:{col}'>"
                     f"{escape(lab)}</span><span>{escape(texto)}</span></div>")
        return html

    def pintar():
        caja = estado["extra"] + juego.eventos
        if not caja:
            caja = [("jefe", f"«{random.choice(FRASES[nivel])}»")]
        frase = (f"<div style='font-style:italic;color:#cbd5e1;margin-top:6px'>{estado['frase']}</div>"
                 if estado["frase"] else "")
        extra = ""
        manual = decidir is None
        if juego.gana:
            if practica:
                extra = ("<div class='ra-pop' style='margin-top:12px;background:#1e3a8a;border-radius:10px;padding:12px'>"
                         "¡Victoria de entrenamiento! La contraseña solo se consigue cuando gana tu código, con luchar().</div>")
            elif nivel == 1:
                extra = ("<div class='ra-pop' style='margin-top:12px;background:#065f46;border-radius:10px;padding:12px;font-size:17px'>"
                         "¡Has ganado a mano! Ahora viene lo importante: que tu robot sepa hacerlo solo. "
                         "Abajo tienes tu forma de jugar convertida en código.</div>")
            elif nivel == NIVEL_SECRETO:
                extra = (f"<div class='ra-pop' style='margin-top:12px;background:#000;border:2px solid #e2e8f0;"
                         f"border-radius:10px;padding:14px;font-size:20px;text-align:center'>LOGRO SECRETO<br>"
                         f"Has vencido a EL ARQUITECTO reescribiendo su mundo.<br>"
                         f"Contraseña secreta: <b style='letter-spacing:3px'>{_clave(NIVEL_SECRETO)}</b><br>"
                         f"<span style='font-size:13px'>No la escribas en la pizarra: díselo a tu profe en voz baja.</span></div>")
            elif nivel == NIVEL_OMEGA:
                extra = ("<div class='ra-pop' style='margin-top:12px;background:#1e3a8a;border-radius:10px;padding:12px'>"
                         "¡Has ganado una partida! Pero OMEGA solo entrega su contraseña a quien gana 70 de 100: "
                         "usa el bucle y retar_a_omega().</div>")
            elif not _limpio():
                extra = ("<div style='margin-top:12px;background:#7f1d1d;border-radius:10px;padding:14px'>"
                         "ANTITRAMPAS: alguien ha cambiado piezas del motor o las reglas desde fuera. "
                         "Esta victoria no da contraseña. Vuelve a ejecutar el motor (Paso 0) y Mi robot.</div>")
            else:
                extra = (f"<div class='ra-pop' style='margin-top:12px;background:#065f46;border-radius:10px;padding:14px;"
                         f"font-size:20px;text-align:center'>¡Has derrotado a {jefe['nombre']}!<br>"
                         f"Contraseña: <b style='letter-spacing:3px'>{_clave(nivel)}</b><br>"
                         f"<span style='font-size:13px'>Escríbela en la pizarra y pasa al siguiente combate.</span></div>")
        if manual and juego.terminado and grab:
            codigo = _cerebro(grab, jefe["nombre"])
            extra += ("<div style='margin-top:12px;background:#0b1220;border:1px solid #334155;border-radius:10px;padding:12px'>"
                      "<b>Tu combate convertido en código</b><br><span style='font-size:13px;color:#cbd5e1'>"
                      "Copia este código en Mi robot, en lugar de la función <code>decidir</code> que haya, "
                      "y vuelve a ejecutar la celda.</span>"
                      f"<textarea readonly rows='{codigo.count(chr(10)) + 2}' style='width:100%;margin-top:8px;background:#111827;"
                      "color:#e2e8f0;border:1px solid #334155;border-radius:6px;font-family:monospace;font-size:13px;padding:8px'>"
                      f"{escape(codigo)}</textarea></div>")
        historial = ""
        fase = f" · FASE {'2' if juego.fase2() else '1'}" if nivel == NIVEL_OMEGA else ""
        modo = " · ENTRENAMIENTO" if practica else (" · LUCHA TU CÓDIGO" if decidir else " · A MANO")
        panel.value = (
            CSS +
            f"<div style='max-width:640px;background:#111827;color:#f8fafc;padding:18px;border-radius:16px;font-family:{FUENTE}'>"
            f"<div style='color:#94a3b8;font-size:12px;letter-spacing:3px'>NIVEL {nivel} · TURNO {juego.turno}{fase}{modo}</div>"
            f"<h2 style='margin:6px 0;color:#f8fafc'>ROBOT ARENA vs {jefe['nombre']}</h2>"
            f"<div style='display:flex;flex-wrap:wrap;background:#1e293b;border-radius:12px'>"
            f"{ficha(str(nombre), mi_color, juego.vida, juego.maxvida, False, estado['d_yo'])}"
            f"<div style='align-self:center;font-size:28px;font-weight:bold;color:#64748b'>VS</div>"
            f"{ficha(jefe['nombre'], COLOR_JEFE[nivel], juego.jefe_vida, juego.jefe_max, True, estado['d_jefe'])}</div>"
            f"<div style='padding:10px 12px;margin-top:12px;background:{'#3f1d1d' if juego.anulada else '#263449'};"
            f"border-radius:8px'>{lineas(caja)}{frase}</div>"
            f"{extra}{historial}"
            f"<div style='margin-top:8px;color:#cbd5e1;font-size:12px'>Turbo: {'disponible' if juego.turbo else 'gastado'}"
            f" · Ataque {juego.ataque} · Curación {juego.curacion}</div></div>")
        for a, b in botones.items():
            b.disabled = juego.terminado or (a == "turbo" and not juego.turbo)

    def jugar_turno(accion):
        antes = (juego.vida, juego.jefe_vida)
        estado["extra"], estado["frase"] = [], ""
        turno = juego.turno
        try:
            juego.actuar(accion)
        except Exception as e:
            juego._sello = juego._foto()
            juego.eventos = [("aviso", _error_codigo(e, "mi_ataque o mi_habilidad"))]
        if juego.turno == turno and not juego.terminado and not juego.eventos:
            if accion == "turbo":
                juego.eventos = [("aviso", "El turbo ya está gastado: solo se puede usar una vez por combate.")]
            elif accion == "habilidad":
                juego.eventos = [("aviso", "La habilidad todavía no funciona en este combate: se desbloquea contra ESPEJO.")]
            else:
                juego.eventos = [("aviso", f"«{accion}» no es una acción válida. Usa: atacar, curar, escudo, turbo o habilidad.")]
        estado["d_yo"] = juego.vida - antes[0]
        estado["d_jefe"] = juego.jefe_vida - antes[1]
        if accion in ("atacar", "turbo", "habilidad") and estilo["frases"] and juego.turno != turno and random.random() < 0.4:
            estado["frase"] = f"{escape(str(nombre))}: «{escape(random.choice(estilo['frases']))}»"
        elif juego.turno != turno and not juego.terminado and random.random() < 0.25:
            estado["frase"] = f"{jefe['nombre']}: «{escape(random.choice(FRASES[nivel]))}»"

    def pulsar(a):
        if a == "siguiente":
            try:
                accion = decidir(juego.info())
            except Exception as e:
                estado["extra"], estado["frase"] = [], ""
                juego.eventos = [("aviso", _error_codigo(e, "decidir"))]
                estado["d_yo"] = estado["d_jefe"] = 0
                pintar()
                return
            if accion == "nada":
                estado["extra"], estado["frase"] = [], ""
                juego.eventos = [("aviso", "Tu robot todavía no tiene cerebro. Gana a CHATARRA a mano con luchar(1), "
                                           "copia en Mi robot el código que te da y vuelve a ejecutar la celda.")]
                estado["d_yo"] = estado["d_jefe"] = 0
                pintar()
                return
            if jefe.get("solo_habilidad"):
                accion = "habilidad"
                jugar_turno(accion)
                estado["extra"] = [("yo", "Contra ESPEJO solo funciona tu habilidad.")] + estado["extra"]
            else:
                jugar_turno(accion)
                estado["extra"] = [("yo", f"Tu código decide: {accion}")] + estado["extra"]
            if juego.historial and juego.historial[-1][0] == juego.turno:
                juego.historial[-1] = (juego.turno, [("yo", f"Decide: {accion}")] + juego.historial[-1][1])
        else:
            antes = {"accion": a, "aviso": juego.aviso(), "vida": juego.vida}
            turno = juego.turno
            jugar_turno(a)
            if juego.turno != turno:
                grab.append(antes)
        pintar()

    for a, b in botones.items():
        b.on_click(lambda _, a=a: pulsar(a))

    def nueva(_):
        juego.reiniciar()
        grab.clear()
        estado.update(d_yo=0, d_jefe=0, extra=[], frase="")
        pintar()
    reset.on_click(nueva)

    pintar()
    display(w.VBox([panel, w.Box(list(botones.values()) + [reset],
                                 layout=w.Layout(display="flex", flex_flow="row wrap", max_width="640px"))]))


def _partida(estrategia, vida, ataque, curacion, regla, habilidad, registro=None):
    c = Combate(vida, ataque, curacion, NIVEL_OMEGA, regla, habilidad)
    while not c.terminado and c.turno < 60:
        accion = estrategia(c.info())
        if accion == "nada":
            raise ValueError("Tu robot todavía no tiene cerebro: gana a CHATARRA a mano con luchar(1) "
                             "y copia en Mi robot el código que te da.")
        turno = c.turno
        c.actuar(accion)
        if c.turno == turno and not c.terminado:
            raise ValueError(f"Tu función decidir ha devuelto {accion!r}, que no se puede usar ahora. Acciones válidas: "
                             "'atacar', 'curar', 'escudo', 'turbo' (solo una vez) y 'habilidad'.")
        if registro is not None:
            registro.append(f"Turno {c.turno} · decide {accion}: {c.mensaje}")
    return c


def _config(*nombres):
    g = globals()
    faltan = [n for n in nombres if n not in g]
    if faltan:
        _aviso("Falta ejecutar alguna celda de Mi robot (no encuentro: " + ", ".join(faltan) + ").")
        return None
    return [g[n] for n in nombres]


def _usando(nombre, vida, ataque, curacion):
    display(HTML(f"<div style='font-family:{FUENTE};color:#64748b;font-size:13px;margin:4px 0'>"
                 f"Usando: {escape(str(nombre))} · vida {vida} · ataque {ataque} · curación {curacion}</div>"))


def _abrir(nivel, practica):
    if nivel not in range(1, NIVEL_OMEGA + 1) and not (nivel == NIVEL_SECRETO and globals().get("_PUERTA_ABIERTA")):
        _aviso("Ese nivel no existe.")
        return
    datos = _config("nombre", "color", "vida", "ataque", "curacion", "mi_ataque", "mi_habilidad", "decidir")
    if datos is None:
        return
    nombre, color, vida, ataque, curacion, regla, habilidad, decidir = datos
    g = globals()
    estilo = {"ojos": g.get("color_ojos", "blanco"), "cabeza": g.get("cabeza", "cuadrada"),
              "antena": g.get("antena", "antena"), "emblema": g.get("emblema", "rayo"), "frases": g.get("frases", [])}
    _usando(nombre, vida, ataque, curacion)
    usa_codigo = nivel >= 2 and not practica
    arena(nombre, color, vida, ataque, curacion, nivel, regla, habilidad, estilo,
          decidir if usa_codigo else None, practica)


def luchar(nivel):
    """Combate definitivo. En el 1 juegas con botones; desde el 2 lucha tu función decidir."""
    _abrir(nivel, practica=False)


def entrenar(nivel):
    """Entrenamiento: juegas a mano contra cualquier jefe. No da contraseña, pero al acabar te enseña tu combate en código."""
    _abrir(nivel, practica=True)


def jugar_partida():
    """Juega UNA partida automática contra OMEGA. Devuelve True si ganas y False si pierdes."""
    datos = _config("decidir", "vida", "ataque", "curacion", "mi_ataque", "mi_habilidad")
    if datos is None:
        raise NameError("Falta ejecutar la celda de Mi robot")
    decidir, vida, ataque, curacion, regla, habilidad = datos
    return _partida(decidir, vida, ataque, curacion, regla, habilidad).gana


def retar_a_omega(partidas=100, *, _check=None):
    """Prueba oficial: OMEGA juega sus propias 100 partidas contra tu código."""
    datos = _config("nombre", "decidir", "vida", "ataque", "curacion", "mi_ataque", "mi_habilidad")
    if datos is None:
        return
    nombre, decidir, vida, ataque, curacion, regla, habilidad = datos
    _usando(nombre, vida, ataque, curacion)
    try:
        Combate(vida, ataque, curacion, NIVEL_OMEGA, regla, habilidad)
    except Trampa as e:
        _aviso(f"ANTITRAMPAS: {escape(str(e))}")
        return
    victorias = anuladas = 0
    registro = []
    for i in range(partidas):
        try:
            c = _partida(decidir, vida, ataque, curacion, regla, habilidad, registro if i == 0 else None)
        except Exception as e:
            _aviso(f"Error en tu código: {escape(type(e).__name__)}: {escape(str(e))}")
            return
        victorias += c.gana
        anuladas += c.anulada
    barra = "█" * (victorias // 5) + "░" * (20 - victorias // 5)
    ok = victorias >= 70
    if ok and not (_check and _check()):
        _aviso("ANTITRAMPAS: alguien ha cambiado piezas del motor o las reglas desde fuera. "
               "No hay contraseña. Vuelve a ejecutar el motor (Paso 0) y Mi robot.")
        return
    texto = (f"<b>Prueba oficial de OMEGA:</b> {victorias} victorias de {partidas}<br>"
             f"<span style='font-family:monospace;font-size:18px'>{barra}</span> (necesitas 70)<br>")
    if anuladas:
        texto += f"{anuladas} partidas anuladas por el ANTITRAMPAS.<br>"
    if ok:
        texto += (f"<div class='ra-pop' style='margin-top:10px;font-size:20px'>¡OMEGA ha caído!<br>"
                  f"Contraseña final: <b style='letter-spacing:3px'>{_clave(NIVEL_OMEGA)}</b></div>")
    else:
        texto += "Todavía no es suficiente. Mira la partida de ejemplo y mejora tu código."
    texto += ("<details style='margin-top:10px'><summary>Ver una partida de ejemplo, turno a turno</summary>"
              "<div style='font-size:12px;line-height:1.5'>" + "<br>".join(escape(r) for r in registro) + "</div></details>")
    _aviso(CSS + texto, "#065f46" if ok else "#1e293b")


def _crear_arbitro():
    g = globals()
    nombres = ["Combate", "Trampa", "validar_stats", "_reglas", "_jefe", "_partida", "_clave", "arena", "_abrir",
               "luchar", "entrenar", "jugar_partida", "retar_a_omega", "_cerebro", "_config", "_rng"]
    piezas = {n: g[n] for n in nombres}
    metodos = dict(g["Combate"].__dict__)
    valores = (g["NIVEL_OMEGA"], g["NIVEL_SECRETO"], dict(g["_K"]))
    defaults = {n: g[n].__defaults__ for n in ("_reglas", "_jefe")}

    def arbitro():
        g = globals()
        if any(g.get(n) is not o for n, o in piezas.items()):
            return False
        actual = piezas["Combate"].__dict__
        if set(actual) != set(metodos) or any(actual[k] is not metodos[k] for k in metodos):
            return False
        if any(piezas[n].__defaults__ != d for n, d in defaults.items()):
            return False
        if {"randint", "random"} & set(vars(piezas["_rng"])):
            return False
        return (g.get("NIVEL_OMEGA"), g.get("NIVEL_SECRETO"), g.get("_K")) == valores
    return arbitro


arena.__kwdefaults__["_check"] = _crear_arbitro()
retar_a_omega.__kwdefaults__["_check"] = arena.__kwdefaults__["_check"]
print("Motor listo. Configura tu robot.")

## Mi robot

### Cómo es

In [ ]:
# Cuando cambies algo, vuelve a ejecutar esta celda.

# Apariencia
nombre = "Fénix"
color = "azul"            # azul, rojo, verde, morado, amarillo, rosa, naranja, blanco, gris, cian o "#ff8800"
color_ojos = "blanco"
cabeza = "cuadrada"       # cuadrada, redonda o triangular
antena = "antena"         # antena, orejas, hélice o nada
emblema = "rayo"          # rayo, estrella, corazón o inicial
frases = ["¡Por la ciencia!", "¡Toma ya!", "Calculando..."]   # dice una al azar cuando ataca

# Atributos: suman 150 y ninguno pasa de 70
# (curacion es la vida que recuperas cada vez que usas "curar")
vida = 50
ataque = 50
curacion = 50

print("Robot listo:", nombre, "· vida", vida, "· ataque", ataque, "· curación", curacion)

### Cómo piensa

In [ ]:
# Cuando cambies algo, vuelve a ejecutar esta celda.

# decidir: el cerebro de tu robot. Cada turno elige qué hacer.
# Acciones: "atacar", "curar", "escudo", "turbo" o "habilidad"
def decidir(robot):
    return "nada"     # tu robot aún no sabe pensar: gana a CHATARRA a mano para conseguir su cerebro

# Ataque: el golpe crítico hace el doble de daño, pero te cuesta 25 de vida
def mi_ataque(dano, robot):
    if robot.vida_jefe < 150:
        dano = dano * 2
    return dano

# Habilidad: se desbloquea contra ESPEJO. Reparte 50 de energía entre daño y curación.
def mi_habilidad(robot):
    dano = 25
    curacion_extra = 25
    return dano, curacion_extra

print("Cerebro cargado.")

## Combates

### 1 · CHATARRA, a mano
**Misión:** Personaliza tu robot y derrota a CHATARRA utilizando los botones. Observa el código que aparece al terminar.  
**Dónde modificar:** *Cómo es*

Un robot de limpieza olvidado en el almacén. Está oxidado, pega flojo y no tiene armadura. Antes de enfrentarte a él, dale a tu robot un aspecto que nadie olvide.

Este combate lo juegas tú con los botones. Tu robot está mirando cómo lo haces.

In [ ]:
luchar(1)

### 2 · CHATARRA, la revancha
**Misión:** Copia el cerebro generado en `decidir`. Consigue que tu robot gane sin que tú elijas sus acciones.  
**Dónde modificar:** *Cómo piensa*

CHATARRA quiere otra oportunidad, pero esta vez no pulsarás ningún botón: luchará el cerebro que tu robot aprendió de ti. Tú solo pulsas «Siguiente turno» para ver qué decide.

Cada turno, tu robot sabe estas cosas y las puede usar para decidir:

| Dato | Qué es |
|---|---|
| `robot.vida`, `robot.vida_max` | Tu vida ahora y tu vida máxima |
| `robot.vida_jefe`, `robot.vida_jefe_max` | La vida del jefe ahora y la máxima |
| `robot.turbo` | `True` si aún no has gastado el turbo |
| `robot.aviso` | Lo que anuncia el jefe: `"nada"`, `"pesado"`, `"cubre"` o `"laser"` |
| `robot.jefe` | El nombre del jefe contra el que luchas |
| `robot.fase` | 1 o 2 (solo en OMEGA) |

In [ ]:
luchar(2)

### 3 · NÉMESIS
**Misión:** Redistribuye tus 150 puntos para superar su armadura de 45.  
**Dónde modificar:** *Cómo es*

El guardián de la vieja fábrica. Su armadura frena 45 puntos de cada golpe, y cuanto más dura el combate, más fuerte pega.

Dicen que solo cae ante robots construidos de otra manera, que golpean más fuerte de lo que su acero aguanta. Si tu robot nunca se cura, quizá los puntos de curación estén mejor en otro sitio.

In [ ]:
entrenar(3)

In [ ]:
luchar(3)

### 4 · MARTILLO
**Misión:** Programa al robot para que utilice el escudo cuando el enemigo anuncie un golpe pesado.  
**Dónde modificar:** `decidir`

Un robot de demolición con armadura 20 y un martillo hidráulico. Casi siempre pega flojo, pero a veces avisa de que prepara un golpe pesado de 60.

Un robot que no escucha los avisos acaba aplastado: cuando MARTILLO prepara el golpe, `robot.aviso` vale `"pesado"`. Si tu cerebro no sabe protegerse, entrena contra MARTILLO y enséñale.

In [ ]:
entrenar(4)

In [ ]:
luchar(4)

### 5 · TITÁN
**Misión:** Programa el ataque crítico para que solo se utilice cuando su núcleo esté desprotegido y tengas suficiente vida.  
**Dónde modificar:** `mi_ataque`

Un gigante de 200 de vida que se repara 8 cada turno. Tu golpe crítico hace el doble de daño, pero te cuesta 25 de vida, y mientras a TITÁN le quede más de la mitad de la vida, su núcleo blindado lo hace rebotar.

Los viejos ingenieros repetían: un buen crítico no depende de la suerte, sino de saber cuándo. Tu robot siempre sabe cuánta vida le queda al jefe: `robot.vida_jefe`.

In [ ]:
entrenar(5)

In [ ]:
luchar(5)

### 6 · ESPEJO
**Misión:** Programa una habilidad que haga daño mientras carga el láser y que te cure el resto del tiempo.  
**Dónde modificar:** `mi_habilidad`

Un robot de cristal: todo el daño que le haces con tus ataques te lo devuelve. Contra él solo funciona tu habilidad, 50 de energía que `mi_habilidad` reparte entre daño directo y curación. Pero ESPEJO también refleja esa energía, salvo mientras carga su láser (cada 3 turnos): entonces su núcleo queda expuesto y recibe el doble.

Una habilidad que hace siempre lo mismo nunca lo romperá. Mientras carga el láser, `robot.aviso` vale `"laser"`.

In [ ]:
entrenar(6)

In [ ]:
luchar(6)

### 7 · OMEGA
**Misión:** Adapta el comportamiento del robot a las dos fases del combate y consigue 70 victorias de 100 partidas.  
**Dónde modificar:** *Cómo piensa* y bucle final

El jefe final: 260 de vida, armadura 40 y dos fases. Mientras le quede más de la mitad de la vida, absorbe el daño de tu habilidad; con la mitad o menos, te devuelve los golpes. Cada 3 turnos dispara un láser de 55. Lo que te sirvió contra ESPEJO quizá no te sirva aquí.

Tu robot sabe en qué fase está (`robot.fase`) y contra quién lucha (`robot.jefe`). OMEGA no se impresiona con una victoria suelta: solo respeta a quien gana una y otra vez. Necesitas ganar 70 de 100 partidas.

In [ ]:
entrenar(7)

In [ ]:
luchar(7)

In [ ]:
victorias = 0
for partida in range(10):      # ojo: este bucle solo juega 10 partidas; para OMEGA cuentan 100
    if jugar_partida():
        victorias = victorias + 1
print("Victorias:", victorias)

Cuando tu bucle diga 70 de 100 o más, pide la prueba oficial:

In [ ]:
retar_a_omega()